In [1]:
import json
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

In [2]:
## creating the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-large-en"
)

C:\Users\ansul\AppData\Local\Temp\ipykernel_17444\1333154181.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3748.80it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECT

In [3]:
## loading the vector store
vectorstore = FAISS.load_local(
    r"..\data\embeddings_semantic_BAAI",
     embedding_model,
     allow_dangerous_deserialization=True
)

In [4]:
## loading eval dataset
with open(r"..\data\retrival_eval.json") as f:
    eval_data = json.load(f)

In [5]:
eval_data[10]

{'query': 'How do changes in currency exchange rates affect international companies?',
 'relevant_chunks': [{'company_name': 'AAPL', 'chunk_id': 19}]}

## Evaluating base bge 

In [6]:
total_queries = len(eval_data)
recall_hits = 0
mrr_total = 0

In [7]:
for item in eval_data:

    query = item["query"]
    relevant = item["relevant_chunks"]

    docs = vectorstore.similarity_search(query, k=20)

    found = False

    for rank, doc in enumerate(docs, start=1):

        meta = doc.metadata

        for rel in relevant:

            if (
                meta["company_name"] == rel["company_name"]
                and meta["chunk_id"] == rel["chunk_id"]
            ):

                if not found:
                    recall_hits += 1
                    mrr_total += 1 / rank
                    found = True

recall_at_k = recall_hits / total_queries
mrr = mrr_total / total_queries

In [8]:
bge_json = {
    "model": "BGE",
    "recall_at_20": recall_at_k,
    "mrr": mrr}

## Evaluating BGE + Max Mariginal Relevance

In [9]:
total_queries = len(eval_data)
recall_hits = 0
mrr_total = 0

In [10]:
for item in eval_data:

    query = item["query"]
    relevant = item["relevant_chunks"]

    docs = vectorstore.max_marginal_relevance_search(query, k=20)

    found = False

    for rank, doc in enumerate(docs, start=1):

        meta = doc.metadata

        for rel in relevant:

            if (
                meta["company_name"] == rel["company_name"]
                and meta["chunk_id"] == rel["chunk_id"]
            ):

                if not found:
                    recall_hits += 1
                    mrr_total += 1 / rank
                    found = True

recall_at_k = recall_hits / total_queries
mrr = mrr_total / total_queries

In [11]:
baai_max_marginal_relevance_json = {
    "model": "BGE and Max Marginal Relevance Search",
    "recall_at_20": recall_at_k,
    "mrr": mrr}

## Evaluating BGE plus Reranking

In [12]:
from sentence_transformers import CrossEncoder
import numpy as np

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

total_queries = len(eval_data)
recall_hits = 0
mrr_total = 0

for item in eval_data:
    query = item["query"]
    relevant = item["relevant_chunks"]

    # first-stage retrieval
    candidates = vectorstore.similarity_search(query, k=20)
    candidate_texts = [doc.page_content for doc in candidates]

    # second-stage reranking using cross-encoder
    pairs = [[query, text] for text in candidate_texts]
    scores = reranker.predict(pairs, convert_to_numpy=True)
    order = np.argsort(scores)[::-1]
    reranked = [candidates[i] for i in order]

    found = False
    for rank, doc in enumerate(reranked, start=1):
        meta = doc.metadata

        for rel in relevant:
            if (
                meta["company_name"] == rel["company_name"]
                and meta["chunk_id"] == rel["chunk_id"]
            ):
                if not found:
                    recall_hits += 1
                    mrr_total += 1 / rank
                    found = True
                break

recall_at_k = recall_hits / total_queries
mrr = mrr_total / total_queries

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1858.05it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
baai_rerank_json = {
    "model": "BGE + CrossEncoder rerank",
    "recall_at_25": recall_at_k,
    "mrr": mrr,
}

In [14]:
# write all results to a JSON file
results = [
    bge_json,
    baai_max_marginal_relevance_json,
    baai_rerank_json,
]
results

[{'model': 'BGE', 'recall_at_20': 0.4, 'mrr': 0.13353174603174603},
 {'model': 'BGE and Max Marginal Relevance Search',
  'recall_at_20': 0.4,
  'mrr': 0.0992478354978355},
 {'model': 'BGE + CrossEncoder rerank',
  'recall_at_25': 0.4,
  'mrr': 0.1934294871794872}]

In [15]:
with open(r"..\data\embeddings_semantic_BAAI\retrival_eval_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


## Summary

The recall is identical for all retrival approaches. However, the MRR score is the best for reranking pipeline which accuractly reflects that reranking is working as expected